<h1>LD Pipeline 2024 (Version 1.0)</h1>

<h3>Verzeichnis bestimmen</h3>

In [ ]:
import os

import pathlib
from pathlib import Path


current_dir = Path.cwd()
print(f"Aktuelles Verzeichnis: {current_dir}")

if os.path.exists(os.path.join(current_dir, "config.ini")):
    print(f"{current_dir} exists")
else:
    parent_dir = current_dir.parent
    if os.path.exists(os.path.join(parent_dir, "config.ini")):
        os.chdir(parent_dir)
        print(f"{parent_dir} neues Verzeichnis")

<h4> Config-Files und Bibliotheken laden<h4>

In [ ]:
import os
import sys
import logging

from pipeline.base import Utils
from pipeline.base import Env, Environment

_e = Env.int
env = Environment(
    _e, [pathlib.Path("config.ini"), pathlib.Path(f"config-{_e.name}.ini")]
)


logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
    stream=sys.stdout,
    force=True,
)

logging.info(f"hallo {env.name}!")

<h3><span style="
    display: inline-block;
    width: 50px;
    height: 50px;
    border-radius: 50%;
    border: 3px solid black;
    font-size: 24px;
    font-weight: bold;
    text-align: center;
    line-height: 50px;
    background-color: yellow;
">
    1
</span> Überprüfe des Start-Signals</h3>
<p style="max-width: 700px;">
    Es wird überprüft, ob im Ordner <b>\\\szh.loc\ssz\applikationen\HDB_Dropzone\PROD\Test\Pipeline</b> ein Start-Signal vorhanden ist. Falls ja, wird das Start-Signal in ein Running-Signal umgewandelt und die Pipeline beginnt mit der Ausführung.
</p>

In [ ]:
import main

utils = Utils()

print(env.name)
print(os.getcwd())
print(env.config.get("start_signal_folder"))
utils.logger.info(f"Pipeline initialized for {env.name}")

if utils.check_start_signal(env):
    print("Start signal found and renamed to a running signal.")
else:
    print("No start signal found.")

<h3><span style="
    display: inline-block;
    width: 50px;
    height: 50px;
    border-radius: 50%;
    border: 3px solid black;
    font-size: 24px;
    font-weight: bold;
    text-align: center;
    line-height: 50px;
    background-color: yellow;
">
    2
</span> Aktualisiere die Pipe-Tabellen</h3>
<p>
    Die relevanten Daten werden aus der HDB in Tabellen mit dem Prefix "pipe_" kopiert, um sie in einen konsistenten Zustand zu archivieren.
</p>

In [ ]:
main.step(name="initPipeTables", env=env)

<h3><span style="
    display: inline-block;
    width: 50px;
    height: 50px;
    border-radius: 50%;
    border: 3px solid black;
    font-size: 24px;
    font-weight: bold;
    text-align: center;
    line-height: 50px;
    background-color: yellow;
">
    3
</span> DB Views aus SQL neu erstellen</h3>

In [ ]:
main.step(name="createViewsFromSQL", env=env)

<h3><span style="
    display: inline-block;
    width: 50px;
    height: 50px;
    border-radius: 50%;
    border: 3px solid black;
    font-size: 24px;
    font-weight: bold;
    text-align: center;
    line-height: 50px;
    background-color: yellow;
">
    4
</span> Generiere die Triple-Dateien</h3>

In [ ]:
triple_types_metadata = [
    "code",
    "cube",
    "groupCode",
    "hierarchy",
    # "legalFoundation",
    "measureUnit",
    "measure",
    "property",
    "room",
    "time",
    "timeTermset",
    "dimensionTermset"
]
triple_types_observations = ["observation"]
triple_types_others = [
    "copyStatic",
    "buildInfo",
    "buildTermsetHierarchy",
    "generateViews",
]

# just for testing
triple_types_metadata = ["dimensionTermset"]
triple_types_observations = []
triple_types_others = []


options_batching = {
    "db_batch_size": 100000,
    "write_batch_size": 600000,
    "max_iteration": None,
}

for name in triple_types_metadata:
    main.step(name=f"{name}Templating", env=env, options=options_batching)
for name in triple_types_observations:
    main.step(name=f"{name}Templating", env=env, options=options_batching)
for name in triple_types_others:
    main.step(name=name, env=env)

<h3><span style="
    display: inline-block;
    width: 50px;
    height: 50px;
    border-radius: 50%;
    border: 3px solid black;
    font-size: 24px;
    font-weight: bold;
    text-align: center;
    line-height: 50px;
    background-color: yellow;
">
    5
</span> Setze das Start-Signal für die Generierung der Fuseki-Indexdateien</h3>

In [ ]:
utils = Utils()
utils.set_start_signal_fuseki_index(env)

<h3><span style="
    display: inline-block;
    width: 50px;
    height: 50px;
    border-radius: 50%;
    border: 3px solid black;
    font-size: 24px;
    font-weight: bold;
    text-align: center;
    line-height: 50px;
    background-color: yellow;
">
    6
</span> Schreibe die Publikationsstati zurück in die HDB</h3>

In [ ]:
main.step(name="writePublicationStatiToHDB", env=env)

<h3><span style="
    display: inline-block;
    width: 50px;
    height: 50px;
    border-radius: 50%;
    border: 3px solid black;
    font-size: 24px;
    font-weight: bold;
    text-align: center;
    line-height: 50px;
    background-color: yellow;
">
    7
</span> Setze das Running-Signal auf Finish</h3>

In [ ]:
utils = Utils()
utils.set_finish_signal(env)

# Extra: Views lokal testen
## Fuseki lokal starten

1. Lokalen fuseki server starten: `docker run --rm -p 3030:3030 --name fuseki stain/jena-fuseki`. Wichtig: Passwort steht im Startup Log.
2. Öffnen des fuseki im Browser unter http://localhost:3030/
3. Benutzername: admin Passwort: aus dem Startup Log

## Views importieren
1. Neues Datenset anlegen (name: `ssz`) (in-memory)
2. Importieren der Views (gzip oder entpackt) unter http://localhost:3030/#/dataset/ssz/upload
3. Unter http://localhost:3030/#/dataset/ssz/query testen, ob views vorhanden sind `SELECT * WHERE { ?a  a <https://cube.link/view/View>} LIMIT 10`

## LD Dereferencer auschecken und starten
1. Git Repo des Dereferenceres clonen: `https://cmp-sdlc.stzh.ch/OE-7035/ssz-lod/ld.stadt-zuerich.ch.git`
2. Wenn nicht vorhanden: installiere node und npm
3. Importiere node_modules: `npm install`
4. Starte lokalen Dereferencer: `npm run start-local`
5. Öffne eine view im Browser z.B. unter http://localhost:8080/statistics/view/BEV411OD411B
6. Lass dir die zugehörigen SPARQL Queries anzeigen, in diesem Fall unter http://localhost:8080/statistics/view/BEV411OD411/Bobservation?format=lsparql

**HINWEIS**: lasse dir alle views im Datenset ssz mit für den lokalen Dereferencer korrekten IRIs listen:
```
SELECT (IRI(REPLACE(STR(?views), "^https://ld.stadt-zuerich.ch", "http://localhost:8080")) AS ?viewsLocal)
WHERE {
  ?views a <https://cube.link/view/View>
}
```

**HINWEIS**: das *.rq file beinhaltet möglicherweise mehrere Zeilen (eine query pro zugehörigen Datacube). Um nur die n-te Zeile anzeigen nutze zB. das terminal (hier die 1. Zeile): `cat BEV411OD411B.rq | sed -n '1 p'`. Für eine schönere Formatierung nutze zB. https://sparql-formatter.dbcls.jp/